In [1]:
# Import all functions from the required modules
from cordo_sherpa_module import *
from cordo_ineris_module import *
from correction_module import *
from expo_functions_module import *
from morbidity_analysis_module import *
from plot_module import *

print("Successfully loaded all modules")

Successfully loaded plotting command
Successfully loaded all modules


In [8]:
#Develop Disability Weights
# Read the excel file
import pandas as pd
import os
import numpy as np

# Read the IHME-GBD data
df = pd.read_excel(os.path.join("data", "1-processed-data", "IHME-GBD_2021_DATA.xlsx"))

# Calculate DW for mean, LCI and UCI values for All ages only
results = []

# Get unique diseases
diseases = df['cause_name'].unique()

# Disease name mapping
disease_mapping = {
    'Diabetes mellitus type 2': "Diabetes T2",
    'Tracheal, bronchus, and lung cancer': "Lung Cancer",
    'Hypertensive heart disease': "Hypertension",
    'Chronic obstructive pulmonary disease': "COPD",
    'Lower respiratory infections': "ALRI in children",
    'Asthma': 'Asthma_adult',
    'Ischemic heart disease': 'IHD events',
    'Stroke': "Stroke"
}

for disease in diseases:
    disease_data = df[(df['cause_name'] == disease) & (df['age_name'] == 'All ages')]

    # Get YLD and Prevalence data for each disease
    yld_data = disease_data[disease_data['measure_name'] == 'YLDs (Years Lived with Disability)']
    prev_data = disease_data[disease_data['measure_name'] == 'Prevalence']

    if not yld_data.empty and not prev_data.empty:
        # Calculate DW for mean values with handling division by zero
        mean_dw = yld_data['val'].values[0] / prev_data['val'].values[0] if prev_data['val'].values[0] != 0 else np.nan

        # Calculate DW for LCI values with handling division by zero
        lci_dw = yld_data['lower'].values[0] / prev_data['lower'].values[0] if prev_data['lower'].values[
                                                                                   0] != 0 else np.nan

        # Calculate DW for UCI values with handling division by zero
        uci_dw = yld_data['upper'].values[0] / prev_data['upper'].values[0] if prev_data['upper'].values[
                                                                                   0] != 0 else np.nan

        results.append({
            'disease': disease_mapping.get(disease, disease),  # Use mapped name if exists, otherwise keep original
            'DW_mean': mean_dw,
            'DW_LCI': lci_dw,
            'DW_UCI': uci_dw
        })

# Convert results to dataframe
dw_df = pd.DataFrame(results)

# Create directory if it doesn't exist
os.makedirs(os.path.join("data", "1-processed-data", "INSEE"), exist_ok=True)

# Save to CSV
output_path = os.path.join("data", "1-processed-data", "INSEE", "disease_disability_weights.csv")
dw_df.to_csv(output_path, index=False)


In [7]:
# Read the excel file
import pandas as pd
import os
import numpy as np

def calculate_dw(disease_name, yld_data, prev_data):
    total_yld = yld_data['val'].sum()
    total_prev = prev_data['val'].sum()
    total_yld_lower = yld_data['lower'].sum()
    total_prev_lower = prev_data['lower'].sum()
    total_yld_upper = yld_data['upper'].sum()
    total_prev_upper = prev_data['upper'].sum()

    mean_dw = total_yld / total_prev if total_prev != 0 else np.nan
    lci_dw = total_yld_lower / total_prev_lower if total_prev_lower != 0 else np.nan
    uci_dw = total_yld_upper / total_prev_upper if total_prev_upper != 0 else np.nan

    return {
        'disease': disease_name,
        'DW_mean': mean_dw,
        'DW_LCI': lci_dw,
        'DW_UCI': uci_dw
    }

try:
    # Read the IHME-GBD data
    df = pd.read_excel(os.path.join("data", "1-processed-data", "IHME-GBD_2021_DATA.xlsx"))

    # Define age ranges for each disease
    disease_age_groups = {
        'Hypertensive heart disease': 'All ages',
        'Tracheal, bronchus, and lung cancer': ['35-39 years', '40-44 years', '45-49 years', '50-54 years',
                                                '55-59 years',
                                                '60-64 years', '65-69 years', '70-74 years', '75-79 years',
                                                '80-84 years',
                                                '85-89 years', '90-94 years', '95+ years'],
        'Asthma': ['<5 years', '5-9 years', '10-14 years', '15-19 years',
                   '20-24 years', '25-29 years', '30-34 years', '35-39 years'],
        'Chronic obstructive pulmonary disease': ['40-44 years', '45-49 years', '50-54 years', '55-59 years',
                                                  '60-64 years',
                                                  '65-69 years', '70-74 years', '75-79 years',
                                                  '80-84 years', '85-89 years', '90-94 years', '95+ years'],
        'Lower respiratory infections': ['<5 years', '5-9 years', '10-14 years'],
        'Stroke': ['35-39 years', '40-44 years', '45-49 years', '50-54 years', '55-59 years', '60-64 years',
                   '65-69 years',
                   '70-74 years', '75-79 years', '80-84 years', '85-89 years', '90-94 years', '95+ years'],
        'Ischemic heart disease': ['35-39 years', '40-44 years', '45-49 years', '50-54 years', '55-59 years',
                                   '60-64 years',
                                   '65-69 years', '70-74 years', '75-79 years', '80-84 years', '85-89 years',
                                   '90-94 years',
                                   '95+ years'],
        'Diabetes mellitus type 2': ['45-49 years', '50-54 years', '55-59 years', '60-64 years', '65-69 years',
                                     '70-74 years', '75-79 years', '80-84 years', '85-89 years', '90-94 years',
                                     '95+ years']
    }

    results = []
    for disease in df['cause_name'].unique():
        if disease in disease_age_groups:
            age_filter = disease_age_groups[disease]

            if age_filter == 'All ages':
                disease_data = df[df['cause_name'] == disease]
            else:  # List of specific age groups
                disease_data = df[(df['cause_name'] == disease) & (df['age_name'].isin(age_filter))]

            # Split asthma data into child and adult age groups
            if disease == 'Asthma':
                # Child age groups
                child_data = disease_data[
                    disease_data['age_name'].isin(['<5 years', '5-9 years', '10-14 years', '15-19 years'])]
                # Adult age groups
                adult_data = disease_data[disease_data['age_name'].isin(
                    ['15-19 years', '20-24 years', '25-29 years', '30-34 years', '35-39 years'])]

                # Process child data
                child_yld = child_data[child_data['measure_name'] == 'YLDs (Years Lived with Disability)']
                child_prev = child_data[child_data['measure_name'] == 'Prevalence']

                # Process adult data
                adult_yld = adult_data[adult_data['measure_name'] == 'YLDs (Years Lived with Disability)']
                adult_prev = adult_data[adult_data['measure_name'] == 'Prevalence']

                # Calculate for children
                if not child_yld.empty and not child_prev.empty:
                    results.append(calculate_dw('Asthma_child', child_yld, child_prev))

                # Calculate for adults
                if not adult_yld.empty and not adult_prev.empty:
                    results.append(calculate_dw('Asthma_adult', adult_yld, adult_prev))

                continue  # Skip the rest of the loop for asthma

            yld_data = disease_data[disease_data['measure_name'] == 'YLDs (Years Lived with Disability)']
            prev_data = disease_data[disease_data['measure_name'] == 'Prevalence']

            if not yld_data.empty and not prev_data.empty:
                total_yld = yld_data['val'].sum()
                total_prev = prev_data['val'].sum()
                total_yld_lower = yld_data['lower'].sum()
                total_prev_lower = prev_data['lower'].sum()
                total_yld_upper = yld_data['upper'].sum()
                total_prev_upper = prev_data['upper'].sum()

                mean_dw = total_yld / total_prev if total_prev != 0 else np.nan
                lci_dw = total_yld_lower / total_prev_lower if total_prev_lower != 0 else np.nan
                uci_dw = total_yld_upper / total_prev_upper if total_prev_upper != 0 else np.nan

                disease_name_mapping = {
                    'Diabetes mellitus type 2': "Diabetes T2",
                    'Tracheal, bronchus, and lung cancer': "Lung Cancer",
                    'Hypertensive heart disease': "Hypertension",
                    'Chronic obstructive pulmonary disease': "COPD",
                    'Lower respiratory infections': "ALRI in children",
                    'Ischemic heart disease': 'IHD events',
                    'Stroke': "Stroke"
                }

                results.append({
                    'disease': disease_name_mapping.get(disease, disease),
                    'DW_mean': mean_dw,
                    'DW_LCI': lci_dw,
                    'DW_UCI': uci_dw
                })

    dw_df = pd.DataFrame(results)
    os.makedirs(os.path.join("data", "1-processed-data", "INSEE"), exist_ok=True)
    output_path = os.path.join("data", "1-processed-data", "INSEE", "disease_disability_weights_age_wise.csv")
    dw_df.to_csv(output_path, index=False)

except Exception as e:
    print(f"Error occurred: {str(e)}")



In [4]:
# Read the excel file
import pandas as pd
import os
import numpy as np

def calculate_dw(disease_name, yld_data, prev_data):
    total_inc = inc_data['val'].sum()
    total_prev = prev_data['val'].sum()
    total_inc_lower = inc_data['lower'].sum()
    total_prev_lower = prev_data['lower'].sum()
    total_inc_upper = inc_data['upper'].sum()
    total_prev_upper = prev_data['upper'].sum()

    mean_dur = total_prev / total_inc if total_inc != 0 else np.nan
    lci_dur = total_prev_lower / total_inc_lower if total_inc_lower != 0 else np.nan
    uci_dur = total_prev_upper/total_inc_upper if total_inc_upper != 0 else np.nan

    return {
        'disease': disease_name,
        'Dur_mean': mean_dur,
        'Dur_LCI': lci_dur,
        'Dur_UCI': uci_dur
    }

try:
    # Read the IHME-GBD data
    df = pd.read_excel(os.path.join("data", "1-processed-data", "IHME-GBD_2021_DATA.xlsx"))

    # Define age ranges for each disease
    disease_age_groups = {
        'Hypertensive heart disease': 'All ages',
        'Tracheal, bronchus, and lung cancer': ['35-39 years', '40-44 years', '45-49 years', '50-54 years',
                                                '55-59 years',
                                                '60-64 years', '65-69 years', '70-74 years', '75-79 years',
                                                '80-84 years',
                                                '85-89 years', '90-94 years', '95+ years'],
        'Asthma': ['<5 years', '5-9 years', '10-14 years', '15-19 years',
                   '20-24 years', '25-29 years', '30-34 years', '35-39 years'],
        'Chronic obstructive pulmonary disease': ['40-44 years', '45-49 years', '50-54 years', '55-59 years',
                                                  '60-64 years',
                                                  '65-69 years', '70-74 years', '75-79 years',
                                                  '80-84 years', '85-89 years', '90-94 years', '95+ years'],
        'Lower respiratory infections': ['<5 years', '5-9 years', '10-14 years'],
        'Stroke': ['35-39 years', '40-44 years', '45-49 years', '50-54 years', '55-59 years', '60-64 years',
                   '65-69 years',
                   '70-74 years', '75-79 years', '80-84 years', '85-89 years', '90-94 years', '95+ years'],
        'Ischemic heart disease': ['35-39 years', '40-44 years', '45-49 years', '50-54 years', '55-59 years',
                                   '60-64 years',
                                   '65-69 years', '70-74 years', '75-79 years', '80-84 years', '85-89 years',
                                   '90-94 years',
                                   '95+ years'],
        'Diabetes mellitus type 2': ['45-49 years', '50-54 years', '55-59 years', '60-64 years', '65-69 years',
                                     '70-74 years', '75-79 years', '80-84 years', '85-89 years', '90-94 years',
                                     '95+ years']
    }

    results = []
    for disease in df['cause_name'].unique():
        if disease in disease_age_groups:
            age_filter = disease_age_groups[disease]

            if age_filter == 'All ages':
                disease_data = df[df['cause_name'] == disease]
            else:  # List of specific age groups
                disease_data = df[(df['cause_name'] == disease) & (df['age_name'].isin(age_filter))]

            # Split asthma data into child and adult age groups
            if disease == 'Asthma':
                # Child age groups
                child_data = disease_data[
                    disease_data['age_name'].isin(['<5 years', '5-9 years', '10-14 years', '15-19 years'])]
                # Adult age groups
                adult_data = disease_data[disease_data['age_name'].isin(
                    ['15-19 years', '20-24 years', '25-29 years', '30-34 years', '35-39 years'])]

                # Process child data
                child_yld = child_data[child_data['measure_name'] == 'YLDs (Years Lived with Disability)']
                child_prev = child_data[child_data['measure_name'] == 'Prevalence']

                # Process adult data
                adult_yld = adult_data[adult_data['measure_name'] == 'YLDs (Years Lived with Disability)']
                adult_prev = adult_data[adult_data['measure_name'] == 'Prevalence']

                # Calculate for children
                if not child_yld.empty and not child_prev.empty:
                    results.append(calculate_dw('Asthma_child', child_yld, child_prev))

                # Calculate for adults
                if not adult_yld.empty and not adult_prev.empty:
                    results.append(calculate_dw('Asthma_adult', adult_yld, adult_prev))

                continue  # Skip the rest of the loop for asthma

            inc_data = disease_data[disease_data['measure_name'] == 'Incidence']
            prev_data = disease_data[disease_data['measure_name'] == 'Prevalence']

            if not inc_data.empty and not prev_data.empty:
                total_inc= inc_data['val'].sum()
                total_prev = prev_data['val'].sum()
                total_inc_lower = inc_data['lower'].sum()
                total_prev_lower = prev_data['lower'].sum()
                total_inc_upper = inc_data['upper'].sum()
                total_prev_upper = prev_data['upper'].sum()

                mean_dur = total_prev/total_inc if total_inc != 0 else np.nan
                lci_dur = total_prev_lower/total_inc_lower if total_inc_lower != 0 else np.nan
                uci_dur= total_prev_upper/total_inc_upper if total_inc_upper != 0 else np.nan

                disease_name_mapping = {
                    'Diabetes mellitus type 2': "Diabetes T2",
                    'Tracheal, bronchus, and lung cancer': "Lung Cancer",
                    'Hypertensive heart disease': "Hypertension",
                    'Chronic obstructive pulmonary disease': "COPD",
                    'Lower respiratory infections': "ALRI in children",
                    'Ischemic heart disease': 'IHD events',
                    'Stroke': "Stroke"
                }

                results.append({
                    'disease': disease_name_mapping.get(disease, disease),
                    'Dur_mean': mean_dur,
                    'Dur_LCI': lci_dur,
                    'Dur_UCI': uci_dur
                })

    dw_df = pd.DataFrame(results)
    os.makedirs(os.path.join("data", "1-processed-data", "INSEE"), exist_ok=True)
    output_path = os.path.join("data", "1-processed-data", "INSEE", "disease_durations_age_wise.csv")
    dw_df.to_csv(output_path, index=False)

except Exception as e:
    print(f"Error occurred: {str(e)}")



In [3]:
# Read the IHME-GBD data
ihme_df = pd.read_excel("data/1-processed-data/IHME-GBD_2021_DATA.xlsx")
# Define morbidity configuration
morb_config = [
    {
        "short": "hypertension", "disease": "hypertensive_heart_disease"
    },
    {
        "short": "ihd", "disease": "ischemic_heart_disease"
    },
    {
        "short": "lung_cancer", "disease": "tracheal,_bronchus,_and_lung_cancer"
    },
    {
        "short": "asthma_child", "disease": "asthma"
    },
    {
        "short": "copd", "disease": "chronic_obstructive_pulmonary_disease"
    },
    {
        "short": "alri", "disease": "lower_respiratory_infections"
    },
    {
        "short": "stroke", "disease": "stroke"
    },
    {
        "short": "ami", "disease": "myocarditis"
    },
    {
        "short": "diabetes_ii", "disease": "diabetes_mellitus_type_2"
    }
]

# Filter for prevalence and incidence measures and All ages
prev_data = ihme_df[(ihme_df['measure_name'] == 'Prevalence') & (ihme_df['age_name'] == 'All ages')].copy()
inc_data = ihme_df[(ihme_df['measure_name'] == 'Incidence') & (ihme_df['age_name'] == 'All ages')].copy()

# Calculate duration for each disease
durations = []
for disease in prev_data['cause_name'].unique():
    # Check if disease exists in both datasets
    prev_vals = prev_data[prev_data['cause_name'] == disease]['val']
    inc_vals = inc_data[inc_data['cause_name'] == disease]['val']

    if not prev_vals.empty and not inc_vals.empty:
        prev = prev_vals.iloc[0]
        inc = inc_vals.iloc[0]

        # Calculate duration in years (prevalence/incidence)
        duration = prev / inc if inc > 0 else float('inf')

        # Get short name from morb_config
        short_name = None
        for config in morb_config:
            if config['disease'].lower().replace(' ', '_') == disease.lower().replace(' ', '_'):
                short_name = config['short']
                break

        # If no short name found, use the original disease name
        if short_name is None:
            short_name = disease.lower().replace(' ', '_')

        durations.append({
            'disease': short_name,
            'prevalence': prev,
            'incidence': inc,
            'duration_years': duration
        })

# Create dataframe with results
duration_df = pd.DataFrame(durations)

# Save as CSV
output_path = os.path.join("data", "1-processed-data", "INSEE")
os.makedirs(output_path, exist_ok=True)
duration_df.to_csv(os.path.join(output_path, "disease_durations.csv"), index=False)
print(duration_df)


        disease    prevalence      incidence  duration_years
0          alri  1.388634e+04  678843.838599        0.020456
1  hypertension  2.346274e+05  387264.000000        0.605859
2           ihd  2.140475e+06  309623.583544        6.913153
3  asthma_child  4.193331e+06  251056.285761       16.702754
4   lung_cancer  1.091752e+05   51462.545149        2.121450
5           ami  5.414429e+03   15324.013614        0.353330
6        stroke  9.147071e+05   94510.463799        9.678368
7          copd  2.919807e+06  203398.319358       14.355119
8   diabetes_ii  3.332599e+06  135938.811007       24.515435


In [ ]:
#code to create a merged file with commune (INSEE) codes, morbidity data and population data
import pandas as pd
import numpy as np
import re

# --- Define file paths ---
health_file = r"C:\Users\aysharma\Eqis_codes\data\Morbidity data check\health_data.xlsx"
population_file = r"C:\Users\aysharma\Eqis_codes\data\Morbidity data check\population_data.xlsx"
output_csv = r"C:\Users\aysharma\Eqis_codes\data\Morbidity data check\final_health_data.csv"

def normalize_insee(val):
    """Normalize INSEE code to 5-character string, preserving Corsica '2A/2B' codes."""
    if pd.isna(val):
        return np.nan
    s = str(val).strip().upper()
    # If looks like a pure float string like "12345.0"
    if re.fullmatch(r"\d+\.0+", s):
        s = s.split(".")[0]
    # Corsica patterns: 2Axxx or 2Bxxx
    if re.fullmatch(r"2[AB]\d{1,3}", s):
        return s[:2] + s[2:].zfill(3)
    # Pure digits -> pad to 5
    if re.fullmatch(r"\d{1,5}", s):
        return s.zfill(5)
    # Already 5+ alphanum, keep first 5
    return (s if len(s) == 5 else (s[:5] if len(s) > 5 else s.zfill(5)))


# --- Step 1: Read all morbidity sheets ---
xl = pd.ExcelFile(health_file)
all_frames = []
for sheet in xl.sheet_names:
    df = pd.read_excel(xl, sheet_name=sheet)
    df["disease"] = sheet
    all_frames.append(df)

morbidity_data = pd.concat(all_frames, ignore_index=True)

# Standardize column names: lower + trim
morbidity_data.columns = [str(c).strip().lower() for c in morbidity_data.columns]

# Normalize insee codes (always 5 characters, handle Corsica)
if "insee" not in morbidity_data.columns:
    raise ValueError("Missing 'insee' column in morbidity data.")
morbidity_data["insee"] = morbidity_data["insee"].map(normalize_insee)

# Collapse potential duplicates: keep first per (insee, disease)
morbidity_data = (
    morbidity_data.sort_index()
    .groupby(["insee", "disease"], as_index=False)
    .first()
)

# Check required columns
if not {"insee", "inc_100000"}.issubset(morbidity_data.columns):
    raise ValueError("Missing 'insee' or 'inc_100000' in morbidity data.")

# --- Step 2: Read population data ---
population_data = pd.read_excel(population_file)
population_data.columns = [str(c).strip().lower() for c in population_data.columns]

if "insee" not in population_data.columns:
    raise ValueError("Missing 'insee' column in population data.")
population_data["insee"] = population_data["insee"].map(normalize_insee)

population_data = (
    population_data.sort_index()
    .groupby("insee", as_index=False)
    .first()
)

# Required population denominators
needed_cols = ["insee", "pop012", "pop1839", "pop017",
               "pop40p", "pop45p", "pop18p", "pop30p", "pop35p"]
missing_pop_cols = [c for c in needed_cols if c not in population_data.columns]
if missing_pop_cols:
    raise ValueError(f"Missing required columns in population data: {missing_pop_cols}")

# --- Step 3: Merge morbidity + population ---
merged_data = pd.merge(
    morbidity_data,
    population_data[needed_cols],
    on="insee",
    how="left",
    validate="m:1"  # Each morbidity row should match at most one population row
)

# Debug: commune counts
unique_insee_morb = morbidity_data["insee"].nunique()
unique_insee_pop = population_data["insee"].nunique()
unique_insee_merged = merged_data["insee"].nunique()
print(f"Unique insee in morbidity_data: {unique_insee_morb}")
print(f"Unique insee in population_data: {unique_insee_pop}")
print(f"Unique insee in merged_data (LEFT join): {unique_insee_merged}")

# Sanity check: INSEE set unchanged after merge
morb_set = set(morbidity_data["insee"].dropna())
merged_set = set(merged_data["insee"].dropna())
if morb_set != merged_set:
    missing_after = morb_set - merged_set
    extra_after = merged_set - morb_set
    raise AssertionError(
        f"INSEE set changed after merge. Missing after: {len(missing_after)}; Extra after: {len(extra_after)}"
    )


# --- Step 4: Compute absolute incidence using appropriate population ---
def pick_denominator(row):
    d = str(row.get("disease", "")).strip()
    if "ALRI in children" in d:
        return row.get("pop012", np.nan)
    if "Asthma in adult" in d:
        return row.get("pop1839", np.nan)
    if "Asthma in children" in d:
        return row.get("pop017", np.nan)
    if "COPD" in d:
        return row.get("pop40p", np.nan)
    if "Stroke" in d:
        return row.get("pop35p", np.nan)
    if "Diabetes T2" in d:
        return row.get("pop45p", np.nan)
    if "Hypertension" in d:
        return row.get("pop18p", np.nan)
    if "IHD events" in d:
        return row.get("pop30p", np.nan)
    if "Lung Cancer" in d:
        return row.get("pop35p", np.nan)
    return np.nan


merged_data["denominator_pop"] = merged_data.apply(pick_denominator, axis=1)

# Ensure numeric
merged_data["inc_100000"] = pd.to_numeric(merged_data["inc_100000"], errors="coerce")
for c in ["pop012", "pop1839", "pop017", "pop40p", "pop45p", "pop18p", "pop30p", "pop35p", "denominator_pop"]:
    if c in merged_data.columns:
        merged_data[c] = pd.to_numeric(merged_data[c], errors="coerce")

merged_data["absolute_incidence"] = merged_data["inc_100000"] * merged_data["denominator_pop"] / 100000.0


# --- Step 5: Consistency check (before vs after totals) ---
def denom_for_check(disease_col):
    d = str(disease_col)
    if "ALRI in children" in d:
        return "pop012"
    if "Asthma in adult" in d:
        return "pop1839"
    if "Asthma in children" in d:
        return "pop017"
    if "COPD" in d:
        return "pop40p"
    if "Stroke" in d:
        return "pop35p"
    if "Diabetes T2" in d:
        return "pop45p"
    if "Hypertension" in d:
        return "pop18p"
    if "IHD events" in d:
        return "pop30p"
    if "Lung Cancer" in d:
        return "pop35p"
    return None


merged_data["_denom_col"] = merged_data["disease"].map(denom_for_check)
before_vals = []
after_vals = []

for dname, grp in merged_data.groupby("disease", dropna=False):
    denom_col = grp["_denom_col"].iloc[0]
    if denom_col is None or denom_col not in grp.columns:
        before = 0.0
    else:
        before = (pd.to_numeric(grp["inc_100000"], errors="coerce") * pd.to_numeric(grp[denom_col],
                                                                                    errors="coerce") / 1e5).sum(
            skipna=True)
    after = pd.to_numeric(grp["absolute_incidence"], errors="coerce").sum(skipna=True)
    before_vals.append(before)
    after_vals.append(after)

check_totals = pd.DataFrame({
    "disease": list(dict.fromkeys(merged_data["disease"])),
    "before": before_vals,
    "after": after_vals
})
check_totals["diff"] = check_totals["after"] - check_totals["before"]
check_totals["rel"] = np.where(check_totals["before"] != 0, 100 * check_totals["diff"] / check_totals["before"], np.nan)
print(check_totals)

# --- Step 6: Export ---
cols_order = list(morbidity_data.columns)
# Place new columns at end if not already present
for extra in ["absolute_incidence"]:
    if extra not in cols_order:
        cols_order.append(extra)

merged_data.to_csv(output_csv, index=False)
print(f"Final data with absolute incidence saved to {output_csv}")


In [ ]:
# Sanity check for the report of SpF
import numpy as np
import pandas as pd

# --- Average national exposure levels (µg/m³) ---
EXPOSURE_PM25 = 9.6
EXPOSURE_NO2 = 8.6

# --- Thresholds (µg/m³) ---
THRESHOLD_PM25 = 3.0
THRESHOLD_NO2 = 1.0

# --- Morbidity configuration ---
# RR values are (central, low, high)
# increment is 10 µg/m³ by default, but can be adjusted if some RR are per 5 µg/m³
morb_config = [
    {"short": "Hypertension", "disease": "Hypertension", "pm25_age_min": 18, "pm25_age_max": 99,
     "rr_ug_PM25_RH50": (1.17, 1.05, 1.30), "rr_ug_NO2": None, "baseline_cases": 708473.75, "increment": 10},
    {"short": "lung_cancer", "disease": "Lung Cancer", "pm25_age_min": 35, "pm25_age_max": 99,
     "rr_ug_PM25_RH50": (1.16, 1.10, 1.23), "rr_ug_NO2": None, "baseline_cases": 39365.40, "increment": 10},
    {"short": "asthma_child", "disease": "Asthma in children", "pm25_age_min": 0, "pm25_age_max": 17,
     "rr_ug_PM25_RH50": (1.34, 1.10, 1.63), "no2_age_min": 0, "no2_age_max": 17,
     "rr_ug_NO2": (1.10, 1.05, 1.18), "baseline_cases": 200984, "increment": 10},
    {"short": "asthma_adult", "disease": "Asthma in adult", "pm25_age_min": 18, "pm25_age_max": 39,
     "rr_ug_PM25_RH50": None, "no2_age_min": 18, "no2_age_max": 39,
     "rr_ug_NO2": (1.10, 1.01, 1.21), "baseline_cases": 91818, "increment": 10},
    {"short": "copd", "disease": "COPD", "pm25_age_min": 40, "pm25_age_max": 99,
     "rr_ug_PM25_RH50": (1.18, 1.13, 1.23), "rr_ug_NO2": None, "baseline_cases": 193327.50, "increment": 10},
    {"short": "alri", "disease": "ALRI in children", "pm25_age_min": 0, "pm25_age_max": 12,
     "rr_ug_PM25_RH50": None, "no2_age_min": 0, "no2_age_max": 12,
     "rr_ug_NO2": (1.09, 1.03, 1.16), "baseline_cases": 57357.75, "increment": 10},
    {"short": "stroke", "disease": "Stroke", "pm25_age_min": 35, "pm25_age_max": 99,
     "rr_ug_PM25_RH50": (1.16, 1.12, 1.20), "rr_ug_NO2": None, "baseline_cases": 96491.75, "increment": 10},
    {"short": "ihd", "disease": "IHD events", "pm25_age_min": 30, "pm25_age_max": 99,
     "rr_ug_PM25_RH50": (1.13, 1.05, 1.22), "rr_ug_NO2": None, "baseline_cases": 95010.50, "increment": 10},
    {"short": "diabetes_2", "disease": "Diabetes T2", "pm25_age_min": 45, "pm25_age_max": 99,
     "rr_ug_PM25_RH50": (1.10, 1.03, 1.18), "rr_ug_NO2": None, "baseline_cases": 209422, "increment": 10}
]


# --- Function to calculate avoided cases ---
def calculate_avoided_cases(RR, baseline_cases, exposure, threshold, increment):
    beta = np.log(RR) / increment
    delta_exposure = max(0, exposure - threshold)
    AF = 1 - np.exp(-beta * delta_exposure)
    avoided_cases = baseline_cases * AF
    return AF, avoided_cases


# --- Loop through diseases and compute avoided cases ---
results = []

for disease in morb_config:
    # Pick pollutant and RRs
    if disease["rr_ug_NO2"] is not None:
        exposure = EXPOSURE_NO2
        threshold = THRESHOLD_NO2
        pollutant = "NO2"
        RR_tuple = disease["rr_ug_NO2"]
    elif disease["rr_ug_PM25_RH50"] is not None:
        exposure = EXPOSURE_PM25
        threshold = THRESHOLD_PM25
        pollutant = "PM2.5"
        RR_tuple = disease["rr_ug_PM25_RH50"]
    else:
        continue

    increment = disease.get("increment", 10)

    # RR_tuple is ordered as (central, low, high). Use correct labels accordingly.
    RR_central, RR_low, RR_high = RR_tuple

    for label, RR in [("Central", RR_central), ("Low", RR_low), ("High", RR_high)]:
        AF, avoided = calculate_avoided_cases(RR, disease["baseline_cases"], exposure, threshold, increment)

        results.append({
            "Disease": disease["disease"],
            "Age group": f"{disease.get('pm25_age_min', disease.get('no2_age_min'))}-{disease.get('pm25_age_max', disease.get('no2_age_max'))}",
            "Pollutant": pollutant,
            "RR type": label,
            "RR": RR,
            "Exposure": exposure,
            "Threshold": threshold,
            "Baseline cases": disease["baseline_cases"],
            "AF": round(AF, 4),
            "Avoided cases": round(avoided, 2)
        })

# --- Output results as DataFrame ---
df_results = pd.DataFrame(results)
print(df_results)


In [ ]:
# Sanity Check of distribution: consider only rows with allocated incidence (>0)
dm = donnees_merged.copy() #upload your age-badsed mortality data
dm["age"] = pd.to_numeric(dm["age"], errors="coerce")
dm["absolute_incidence_iris"] = pd.to_numeric(dm.get("absolute_incidence_iris", 0.0), errors="coerce").fillna(0.0)

rows = []
violating_ages = {}

for disease, (amin, amax) in disease_age_groups.items():
    # only rows that actually received allocated incidence
    sub = dm.loc[(dm["disease"] == disease) & (dm["absolute_incidence_iris"] > 0)].copy()
    n_rows = len(sub)
    if n_rows == 0:
        rows.append({
            "disease": disease,
            "age_min_expected": amin,
            "age_max_expected": amax,
            "n_rows": 0,
            "age_min_present": None,
            "age_max_present": None,
            "n_outside_range": 0,
            "abs_inc_outside": 0.0,
            "abs_inc_total": 0.0,
            "ok": True,   # no allocated rows -> considered OK
            "note": "no allocated incidence rows"
        })
        violating_ages[disease] = []
        continue

    sub = sub.dropna(subset=["age"])
    if sub.empty:
        rows.append({
            "disease": disease,
            "age_min_expected": amin,
            "age_max_expected": amax,
            "n_rows": n_rows,
            "age_min_present": None,
            "age_max_present": None,
            "n_outside_range": 0,
            "abs_inc_outside": 0.0,
            "abs_inc_total": float(dm.loc[dm["disease"] == disease, "absolute_incidence_iris"].sum()),
            "ok": False,
            "note": "allocated rows exist but ages are NaN"
        })
        violating_ages[disease] = []
        continue

    age_min_present = int(sub["age"].min())
    age_max_present = int(sub["age"].max())
    outside_mask = (sub["age"] < amin) | (sub["age"] > amax)
    n_outside = int(outside_mask.sum())
    abs_inc_outside = float(sub.loc[outside_mask, "absolute_incidence_iris"].sum())
    abs_inc_total = float(sub["absolute_incidence_iris"].sum())

    ok_flag = (n_outside == 0)
    violating_ages[disease] = sorted(sub.loc[outside_mask, "age"].unique().tolist()) if n_outside > 0 else []

    rows.append({
        "disease": disease,
        "age_min_expected": amin,
        "age_max_expected": amax,
        "n_rows": n_rows,
        "age_min_present": age_min_present,
        "age_max_present": age_max_present,
        "n_outside_range": n_outside,
        "abs_inc_outside": abs_inc_outside,
        "abs_inc_total": abs_inc_total,
        "ok": bool(ok_flag),
        "note": "" if ok_flag else "allocated incidence outside expected age range"
    })

validation_summary = pd.DataFrame(rows).sort_values(["ok", "disease"], ascending=[True, True]).reset_index(drop=True)
age_group_check_passed = bool((validation_summary["n_outside_range"] == 0).all())

print(validation_summary)
print(f"\nAge group validation passed for all diseases (allocated rows only): {age_group_check_passed}")
for d, ages in violating_ages.items():
    if ages:
        print(f" - {d}: {ages}")

# Optional: explicitly clear 'disease' labels on rows outside each disease age range
# (prevents downstream code from treating those rows as belonging to the disease)
for disease, (amin, amax) in disease_age_groups.items():
    outside = (donnees_merged["disease"] == disease) & ~donnees_merged["age"].between(amin, amax)
    donnees_merged.loc[outside, "disease"] = np.nan